# Phase 0 — Infrastructure Validation

This notebook proves the Phase 0 stack is healthy before we build on it — mirroring how the reference course uses `notebooks/` to validate each week's services.

**Prerequisites**
1. `cp .env.example .env` and set `ANTHROPIC_API_KEY` (the Claude check is skipped if it's empty).
2. `uv sync --extra dev`
3. `docker compose up -d` (Postgres + OpenSearch + Redis) — wait until healthy.

Run all cells. Each check sets an `ok_*` flag; the final cell prints a summary.

## Setup
Locate the repo root (so `.env` loads and `archaeologist` imports), then print the resolved config.

In [ ]:
import os, sys
from pathlib import Path

# Walk up to the dir containing pyproject.toml, chdir there, expose src/ for imports.
root = Path.cwd()
while root != root.parent and not (root / "pyproject.toml").exists():
    root = root.parent
os.chdir(root)
sys.path.insert(0, str(root / "src"))

from archaeologist.config import settings

print("repo root      :", root)
print("env            :", settings.app_env)
print("postgres       :", f"{settings.postgres_host}:{settings.postgres_port}/{settings.postgres_db}")
print("opensearch     :", settings.opensearch_url)
print("redis          :", settings.redis_url)
print("claude model   :", settings.claude_model)
print("anthropic key  :", "set" if settings.anthropic_api_key else "MISSING")

## 1. PostgreSQL
Connect and read the server version.

In [ ]:
import psycopg

ok_postgres = False
try:
    with psycopg.connect(
        host=settings.postgres_host, port=settings.postgres_port,
        user=settings.postgres_user, password=settings.postgres_password,
        dbname=settings.postgres_db, connect_timeout=3,
    ) as conn:
        version = conn.execute("SELECT version()").fetchone()[0]
    ok_postgres = True
    print("OK:", version)
except Exception as exc:
    print("FAILED:", exc)

## 2. OpenSearch
Query cluster health (should be `green` or `yellow` for a single node).

In [ ]:
import httpx

ok_opensearch = False
try:
    resp = httpx.get(f"{settings.opensearch_url}/_cluster/health", timeout=5.0)
    resp.raise_for_status()
    health = resp.json()
    ok_opensearch = health["status"] in ("green", "yellow")
    print("OK: status =", health["status"], "| nodes =", health["number_of_nodes"])
except Exception as exc:
    print("FAILED:", exc)

## 3. Redis
Ping, then a set/get roundtrip.

In [ ]:
import redis

ok_redis = False
try:
    client = redis.Redis(host=settings.redis_host, port=settings.redis_port, socket_timeout=3)
    client.set("arch:phase0", "hello")
    value = client.get("arch:phase0")
    ok_redis = client.ping() and value == b"hello"
    print("OK: ping + set/get roundtrip ->", value)
except Exception as exc:
    print("FAILED:", exc)

## 4. Claude API (optional)
A tiny round-trip to confirm the key + model work. Skipped if `ANTHROPIC_API_KEY` is empty. Costs a few tokens.

In [ ]:
ok_claude = None  # None == skipped
if not settings.anthropic_api_key:
    print("SKIPPED: ANTHROPIC_API_KEY not set in .env")
else:
    try:
        import anthropic
        client = anthropic.Anthropic(api_key=settings.anthropic_api_key)
        msg = client.messages.create(
            model=settings.claude_model,
            max_tokens=16,
            messages=[{"role": "user", "content": "Reply with the single word: pong"}],
        )
        ok_claude = True
        print("OK:", settings.claude_model, "->", msg.content[0].text.strip())
    except Exception as exc:
        ok_claude = False
        print("FAILED:", exc)

## 5. FastAPI app (in-process)
Exercise the app via `TestClient` — no running server needed. `/health/deps` re-checks all three services through the app's own code path.

In [ ]:
import json
from fastapi.testclient import TestClient
from archaeologist.main import app

test_client = TestClient(app)
print("/health      ->", test_client.get("/health").json())
deps = test_client.get("/health/deps").json()
print("/health/deps ->", json.dumps(deps, indent=2))

## Summary

In [ ]:
def badge(flag):
    return "SKIPPED" if flag is None else ("PASS" if flag else "FAIL")

print("Phase 0 — infrastructure validation")
print("  PostgreSQL :", badge(ok_postgres))
print("  OpenSearch :", badge(ok_opensearch))
print("  Redis      :", badge(ok_redis))
print("  Claude API :", badge(ok_claude))

core_ok = ok_postgres and ok_opensearch and ok_redis
print()
print("Core infra:", "READY ✅" if core_ok else "NOT READY ❌ (is `docker compose up -d` finished?)")